# LERF-AA — LERF Features for Authorship Attribution (Ch.7)

The third and final method: turn the Ch.6 LERF estimator into a
*feature extractor* for standard supervised classifiers.

**The idea.** Represent each document by its **LERF profile** — the
50,257-dim vocabulary-wide distribution a *frozen* GPT-2 expects, given
that document's contexts — and let ordinary classifiers learn which
profiles belong to which author. No fine-tuning anywhere; one shared GPT-2
processes every document.

**Realised vs. expected language (Ch.8 framing).** Where ALMs ask "how
predictable are the *actual tokens* of the questioned document under each
author's model?" (modelling *realised* language), LERF-AA asks "what does
the *expected vocabulary-wide distribution* of this document look like?"
(modelling *expected* language). The two views are complementary — the
thesis reports both on the same benchmarks.

**The pipeline (Ch.7 Sec 7.3):**
1. **Feature extraction** — `extract_lerf_features`: one LERF profile per
   document (`(n_docs, 50257)` matrix; each row sums to 1).
2. **MFW selection** — `select_mfw`: keep only the top-*k* types by
   *training-set* frequency (leak-free: test data never influences the
   ranking).
3. **Classification** — `run_lerf_aa` / `full_pipeline`: fit the 8 thesis
   classifiers, report macro-accuracy, top-N accuracy and true-author rank
   stats.

## 0. Bootstrap

In [1]:
import os, sys

REPO_ROOT = os.path.dirname(os.path.abspath(os.getcwd()))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd

from thesis_aa import config, data as data_mod
from thesis_aa.lerf import lerf_aa

print('device:', config.get_device())

C:\Users\MiraMoe\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: xpu


## 1. Load a corpus

We use the synthetic corpus with 5 authors — enough classes for the
ranking metrics (top-1..top-5) to be meaningful, still tiny enough for CPU
feature extraction.

In [2]:
train_df, test_df = data_mod.generate_synthetic(
    n_authors=5, n_train_docs=10, n_test_docs=4, max_words=60, seed=0)
print('train:', train_df.shape, '| test:', test_df.shape)
print(train_df['author_tag'].value_counts())

train: (50, 2) | test: (20, 2)
author_tag
author00    10
author01    10
author02    10
author03    10
author04    10
Name: count, dtype: int64


## 2. Step 1 — Extract LERF features

Each document is treated as an independent "sample corpus" and fed through
`lerf_estimate` (the Ch.6 machinery): for every context position the frozen
GPT-2 emits a full next-token distribution, and those distributions are
summed and normalised into one 50,257-dim profile. This is the only
compute-heavy step — one forward pass per (document, position-window).

In [3]:
X_train = lerf_aa.extract_lerf_features(
    train_df, model_name='gpt2', device=config.get_device())
X_test = lerf_aa.extract_lerf_features(
    test_df, model_name='gpt2', device=config.get_device())

print('X_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)
print('row sums (should be 1.0):', X_train.sum(axis=1)[:4])

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.9.0+xpu).


W0901 02:33:59.454000 15664 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


X_train shape: (50, 50257)
X_test shape : (20, 50257)
row sums (should be 1.0): [1. 1. 1. 1.]


**What you should see:** `(50, 50257)` and `(20, 50257)` matrices
whose rows each sum to 1 — every document is now a probability
distribution over GPT-2's vocabulary. Conceptually, each row is "the
expected language of one document" — 50,257 LLM-informed stylometric
features per document.

In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('gpt2')

# Peek at one document's profile: its top tokens by expected frequency.
doc0 = X_train[0]
top = np.argsort(doc0)[::-1][:10]
pd.DataFrame({
    'token': [tokenizer.decode([i]) for i in top],
    'LERF p': [f'{doc0[i]:.4f}' for i in top],
})

,token,LERF p
0,or,0.0488
1,of,0.0220
2,the,0.0195
3,our,0.0184
4,knight,0.0173
5,in,0.0170
6,was,0.0159
7,",",0.0156
8,he,0.0140
9,she,0.0136


**What you should see:** the profile's top tokens for the first
training document — dominated by function words (genuinely expected in any
English context), plus a tail of thousands of low-probability types that
nevertheless differ subtly between authors. It is exactly that *tail*
structure — the expected frequencies of words the document never uses —
that carries the authorship signal.

## 3. Step 2 — MFW selection

The full 50,257-dim profile is more than many classifiers need. MFW
("most frequent words") selection keeps the *k* vocabulary types most
frequent in the *training* data. The ranking uses training data only —
using test data would leak test information into feature selection and
inflate accuracy. The selected *indices* are then applied unchanged to
both matrices.

In [5]:
import numpy as np

k = 50
Xtr_sel, Xte_sel, indices = lerf_aa.select_mfw(
    X_train, X_test, train_df['text'].tolist(), k, tokenizer=tokenizer)

print('selected shape:', Xtr_sel.shape)
print()
print('top-20 selected tokens (by training frequency):')
print(' ', [tokenizer.decode([i]) for i in indices[:20]])

selected shape: (50, 50)

top-20 selected tokens (by training frequency):
  [' on', ' they', ' in', ' his', ' a', ' were', ' she', ' by', ' it', 'al', ' as', ' and', ' the', ' he', ' that', ' not', ' all', ' with', ' at', ' me']


**What you should see:** the ids and strings of the 50
training-most-frequent tokens — on this synthetic corpus mostly shared
function words (the generator's ~30-word common set), possibly with a few
author-lexicon fragments mixed in. The same column indices slice both
`X_train` and `X_test`.

## 4. Step 3 — Classify with the 8 thesis classifiers

`build_classifiers` instantiates the exact Ch.7 Sec 7.3.3 configurations —
chosen to span classifier families (linear-margin, probabilistic, tree
ensembles, boosting) and to exclude methods whose cost scales badly at
50k dims (libsvm SVC, k-NN, deep MLPs):

| Classifier | Configuration |
|---|---|
| Linear SVM | `SGDClassifier(hinge)`, O(n) — the thesis's headline classifier |
| Logistic Regression | `SGDClassifier(log_loss)` |
| Random Forest | 100 trees, `sqrt` features |
| Extra Trees | 100 trees, depth 50 |
| Gaussian Naive Bayes | default (motivated: LERF values are probabilities) |
| Decision Tree | depth 50 |
| AdaBoost | 30 stumps |
| Histogram GB | 30 iters, depth 3 |

We run them on the k=50 features via `run_lerf_aa`, which also builds
ranked candidate lists per document (decision function → predict_proba →
one-hot fallback) for the top-N metrics:

In [6]:
y_train = train_df['author_tag'].to_numpy()
y_test = test_df['author_tag'].to_numpy()

results_k50 = lerf_aa.run_lerf_aa(Xtr_sel, y_train, Xte_sel, y_test)
cols = ['classifier', 'macro_accuracy', 'top_1', 'top_5',
        'rank_mean', 'rank_Q50']
display(results_k50[cols].sort_values('macro_accuracy', ascending=False))

,classifier,macro_accuracy,top_1,top_5,rank_mean,rank_Q50
3,Extra Trees,1.00,1.00,1.0,1.00,1.0
2,Random Forest,1.00,1.00,1.0,1.00,1.0
4,Gaussian Naive Bayes,1.00,1.00,1.0,1.00,1.0
6,AdaBoost,0.95,0.95,1.0,1.05,1.0
7,Histogram Gradient Boosting,0.95,0.95,1.0,1.05,1.0
5,Decision Tree,0.90,0.90,1.0,1.25,1.0
0,Linear SVM,0.60,0.60,1.0,2.40,1.0
1,Logistic Regression,0.20,0.20,1.0,2.50,2.0


**What you should see:** one row per classifier with macro-accuracy,
top-1/top-5 accuracy and true-author rank statistics (mean, median, Q99 —
"how far down the ranked list is the true author, on average"). On the
synthetic corpus with distinct author lexicons, most classifiers reach
near-perfect accuracy; the interesting thesis question is whether that
holds at 50 candidates and full vocabulary on real data — there Linear
SVM wins with 79.2% mean macro-accuracy.

## 5. The full pipeline across MFW sizes

`full_pipeline` extracts features once, then sweeps the MFW sizes and
writes one CSV per size to `results/`. The thesis evaluates seven sizes:
50, 100, 150, 200, 500, 1000, and the full 50,257-type vocabulary
(`config.MFW_SIZES`). We sweep the first two plus full-vocab to keep the
demo fast:

In [7]:
results = lerf_aa.full_pipeline(
    train_df, test_df, model_name='gpt2',
    device=config.get_device(),
    mfw_sizes=[50, 100, 50257],   # thesis: config.MFW_SIZES (all seven)
)

pivot = pd.concat(results.values())[ ['mfw_size', 'classifier', 'macro_accuracy'] ]
pivot = pivot.pivot(index='classifier', columns='mfw_size', values='macro_accuracy')
pivot = pivot[[50, 100, 50257]].rename(columns={50257: 'full'})
display(pivot.round(3).sort_values(50, ascending=False))

[LERF-AA] MFW=50 -> D:\AgentHome\Thesis\results\lerf_aa_mfw-50.csv


[LERF-AA] MFW=100 -> D:\AgentHome\Thesis\results\lerf_aa_mfw-100.csv


[LERF-AA] MFW=50257 -> D:\AgentHome\Thesis\results\lerf_aa_mfw-50257.csv


mfw_size,50,100,full
classifier,,,
Gaussian Naive Bayes,1.00,1.0,1.00
Extra Trees,1.00,1.0,1.00
Random Forest,1.00,1.0,1.00
AdaBoost,0.95,1.0,1.00
Histogram Gradient Boosting,0.95,1.0,1.00
Decision Tree,0.80,1.0,0.95
Linear SVM,0.45,0.2,1.00
Logistic Regression,0.20,0.2,0.20


**What you should see:** a classifier × feature-size accuracy table.
Watch how accuracy responds to *k* — the thesis (Ch.7 Sec 7.4) finds the
MFW sizes plateau early (a few hundred types capture most signal) but the
best overall config is the **full 50,257-dim vocabulary with Linear SVM**,
which is why that pair is the LERF-AA headline. Per-size CSVs are written
under `results/lerf_aa_mfw-<k>.csv` with all 13 metric columns.

One caveat visible in this demo: the SGD-based linear models look weak and
unstable here because they take stochastic gradient passes over only 50
training documents — on the real benchmarks (thousands of documents per
author) they are the strongest performers, which is exactly why Linear SVM
is the thesis headline.

In [8]:
import glob
print('artifacts written by full_pipeline:')
for f in sorted(glob.glob(os.path.join(config.RESULTS_DIR, 'lerf_aa_mfw-*.csv'))):
    print(' ', os.path.basename(f))

artifacts written by full_pipeline:
  lerf_aa_mfw-100.csv
  lerf_aa_mfw-50.csv
  lerf_aa_mfw-50257.csv


## 6. Going to real data

```python
train_df, test_df = data_mod.load_benchmark('Blogs50')   # 50 candidates
results = lerf_aa.full_pipeline(
    train_df, test_df,
    model_name='gpt2-xl',            # thesis model (1.5B)
    device=config.get_device(),
    mfw_sizes=config.MFW_SIZES,      # all seven sizes
)
```

| Aspect | Demo | Thesis |
|---|---|---|
| Candidates | 5 synthetic authors | 50 real authors |
| Model | `gpt2` | `gpt2` base → `gpt2-xl` (best) |
| MFW sizes | {50, 100, full} | {50, 100, 150, 200, 500, 1000, full} |
| Headline | ~perfect (synthetic data is easy) | 79.2% mean macro-accuracy (Linear SVM, full vocab) |

No training of the LLM happens anywhere in LERF-AA — feature extraction
(one forward pass per document window) dominates the runtime, so scaling
up means a bigger *frozen* model, not longer training. Together with
`ALMs_Train.ipynb` / `ALMs_PPL.ipynb` (realised language) and
`LERF_Estimate.ipynb` (the estimator itself), this completes the tour of
all three thesis methods.